# 3D Reconstruction + Multimodal QA Pipeline

This notebook demonstrates the complete pipeline for:
1. Multi-view 3D reconstruction
2. Instance segmentation with masks
3. Multimodal QA using LMM (Qwen2-VL)

**Target GPU**: T4 or P100 (12-16GB VRAM)

**Note**: This notebook prioritizes demo reliability over cutting-edge accuracy.

## Cell 1: Setup & Dependencies

Install required packages and check GPU availability.

In [ ]:
# Install required packages
!pip install -q transformers>=4.36.0 accelerate>=0.25.0 bitsandbytes>=0.41.0
!pip install -q torch torchvision
!pip install -q open3d>=0.18.0
!pip install -q Pillow opencv-python-headless
!pip install -q scipy numpy matplotlib
!pip install -q flask

# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU available. Running on CPU will be slow.")

## Cell 2: Configuration

Define paths, model names, and quality settings.

In [ ]:
import os
from pathlib import Path
from dataclasses import dataclass
from typing import Literal

@dataclass
class Config:
    """Pipeline configuration."""
    # Paths
    input_dir: str = "./input_images"
    output_dir: str = "./output"
    cache_dir: str = "/root/.cache/huggingface"
    
    # Model settings
    lmm_model: str = "Qwen/Qwen2-VL-2B-Instruct"  # Use 2B for T4
    # lmm_model: str = "Qwen/Qwen2-VL-7B-Instruct"  # Use 7B for P100 with more VRAM
    quantization: Literal["none", "4bit", "8bit"] = "4bit"
    
    # Reconstruction settings
    mode: Literal["quick", "full"] = "quick"
    quick_iterations: int = 100
    full_iterations: int = 500
    
    # Image settings
    max_image_size: int = 1024
    
    # Demo settings
    use_fallback: bool = True  # Use fallback if processing fails

config = Config()

# Create output directories
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(f"{config.output_dir}/instances", exist_ok=True)
os.makedirs(config.input_dir, exist_ok=True)

print(f"Configuration:")
print(f"  Mode: {config.mode}")
print(f"  LMM Model: {config.lmm_model}")
print(f"  Quantization: {config.quantization}")
print(f"  Output: {config.output_dir}")

## Cell 3: Load Models

Load the quantized LMM for multimodal QA. Uses 4-bit quantization to fit on T4/P100.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from transformers import BitsAndBytesConfig
import torch

def load_lmm_model(config):
    """Load the multimodal LLM with quantization."""
    print(f"Loading {config.lmm_model}...")
    
    # Configure quantization
    if config.quantization == "4bit":
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
    elif config.quantization == "8bit":
        quantization_config = BitsAndBytesConfig(
            load_in_8bit=True
        )
    else:
        quantization_config = None
    
    try:
        # Load model
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            config.lmm_model,
            quantization_config=quantization_config,
            device_map="auto",
            torch_dtype=torch.float16,
            cache_dir=config.cache_dir,
            trust_remote_code=True,
        )
        
        # Load processor
        processor = AutoProcessor.from_pretrained(
            config.lmm_model,
            cache_dir=config.cache_dir,
            trust_remote_code=True,
        )
        
        print(f"Model loaded successfully!")
        print(f"Memory usage: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
        
        return model, processor
        
    except Exception as e:
        print(f"Failed to load model: {e}")
        print("Falling back to mock inference mode.")
        return None, None

# Load models (or set to None for demo mode)
try:
    lmm_model, lmm_processor = load_lmm_model(config)
except Exception as e:
    print(f"Model loading skipped: {e}")
    lmm_model, lmm_processor = None, None

## Cell 4: Reconstruction Pipeline Stub

Placeholder for IGGT or alternative 3D reconstruction. For demo, generates mock data.

In [ ]:
import numpy as np
import json
from PIL import Image
import cv2

def run_reconstruction(image_paths, config):
    """
    Run 3D reconstruction pipeline.
    
    In production, this would:
    1. Extract features from images
    2. Estimate camera poses
    3. Generate dense point cloud
    
    For demo, generates placeholder data.
    """
    print(f"Running {config.mode} reconstruction on {len(image_paths)} images...")
    
    # Placeholder: Generate mock camera poses
    num_cameras = len(image_paths) if image_paths else 12
    poses = []
    for i in range(num_cameras):
        angle = (i / num_cameras) * 2 * np.pi
        poses.append({
            "id": i,
            "position": [
                float(3 * np.cos(angle)),
                1.5,
                float(3 * np.sin(angle))
            ],
            "rotation": [0, float(np.degrees(angle)), 0],
            "image": image_paths[i] if i < len(image_paths) else f"image_{i:03d}.jpg"
        })
    
    # Save poses
    poses_path = f"{config.output_dir}/poses.json"
    with open(poses_path, "w") as f:
        json.dump({"cameras": poses}, f, indent=2)
    print(f"Saved camera poses to {poses_path}")
    
    # Placeholder: Generate mock point cloud
    num_points = 50000 if config.mode == "quick" else 200000
    points = np.random.randn(num_points, 3) * np.array([2.5, 1.5, 2.5])
    points[:, 1] = np.abs(points[:, 1])  # Keep Y positive
    
    # Generate colors (warm tones for living room)
    colors = np.random.rand(num_points, 3) * np.array([0.4, 0.3, 0.2]) + np.array([0.6, 0.5, 0.4])
    colors = (colors * 255).astype(np.uint8)
    
    # Save as PLY
    ply_path = f"{config.output_dir}/pointcloud.ply"
    save_ply(ply_path, points, colors)
    print(f"Saved point cloud ({num_points:,} points) to {ply_path}")
    
    return {
        "poses_path": poses_path,
        "pointcloud_path": ply_path,
        "num_cameras": num_cameras,
        "num_points": num_points
    }

def save_ply(path, points, colors):
    """Save point cloud as PLY file."""
    with open(path, "w") as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {len(points)}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        f.write("property uchar red\n")
        f.write("property uchar green\n")
        f.write("property uchar blue\n")
        f.write("end_header\n")
        for i in range(len(points)):
            f.write(f"{points[i, 0]:.6f} {points[i, 1]:.6f} {points[i, 2]:.6f} ")
            f.write(f"{colors[i, 0]} {colors[i, 1]} {colors[i, 2]}\n")

# Run reconstruction with sample/empty images
sample_images = list(Path(config.input_dir).glob("*.jpg")) + list(Path(config.input_dir).glob("*.png"))
reconstruction_result = run_reconstruction([str(p) for p in sample_images], config)
print(f"\nReconstruction complete: {reconstruction_result}")

## Cell 5: Instance Segmentation Stub

Placeholder for SAM or similar instance segmentation. Generates mock masks.

In [ ]:
import numpy as np
from PIL import Image
import os

def run_instance_segmentation(config):
    """
    Run instance segmentation.
    
    In production, this would:
    1. Load SAM or similar model
    2. Generate instance masks for each object
    3. Compute embeddings for each instance
    
    For demo, generates placeholder masks.
    """
    print("Running instance segmentation...")
    
    # Define mock objects
    objects = [
        {"id": "obj_001", "class": "sofa", "position": [2.35, 0.45, 2.65], "dimensions": [2.3, 0.9, 1.1]},
        {"id": "obj_002", "class": "coffee_table", "position": [2.5, 0.25, 1.5], "dimensions": [1.2, 0.5, 0.6]},
        {"id": "obj_003", "class": "armchair", "position": [0.5, 0.4, 2.0], "dimensions": [0.9, 0.8, 0.9]},
        {"id": "obj_004", "class": "armchair", "position": [4.2, 0.4, 2.0], "dimensions": [0.9, 0.8, 0.9]},
        {"id": "obj_005", "class": "floor_lamp", "position": [0.3, 0.75, 3.5], "dimensions": [0.3, 1.5, 0.3]},
        {"id": "obj_006", "class": "bookshelf", "position": [4.5, 1.0, 4.0], "dimensions": [1.0, 2.0, 0.4]},
        {"id": "obj_007", "class": "rug", "position": [2.5, 0.01, 2.0], "dimensions": [3.0, 0.02, 2.0]},
        {"id": "obj_008", "class": "tv_stand", "position": [2.5, 0.3, 0.3], "dimensions": [1.8, 0.6, 0.4]},
    ]
    
    instances_dir = f"{config.output_dir}/instances"
    os.makedirs(instances_dir, exist_ok=True)
    
    # Generate mock masks (simple colored rectangles)
    mask_size = (256, 256)
    for obj in objects:
        # Create a simple mask
        mask = np.zeros(mask_size, dtype=np.uint8)
        # Random ellipse to simulate object shape
        center = (np.random.randint(64, 192), np.random.randint(64, 192))
        axes = (np.random.randint(30, 80), np.random.randint(30, 80))
        cv2.ellipse(mask, center, axes, 0, 0, 360, 255, -1)
        
        # Save mask
        mask_path = f"{instances_dir}/{obj['id']}_mask.png"
        Image.fromarray(mask).save(mask_path)
        obj["mask_path"] = f"instances/{obj['id']}_mask.png"
        
        # Generate mock embedding (128-dim)
        obj["embedding"] = np.random.randn(128).tolist()
    
    print(f"Generated {len(objects)} instance masks")
    return objects

instances = run_instance_segmentation(config)
print(f"\nDetected instances: {[obj['class'] for obj in instances]}")

## Cell 6: Scene Structuring

Convert reconstruction output to structured scene.json format.

In [ ]:
import json

def generate_scene_structure(instances, config):
    """
    Generate structured scene.json from reconstruction data.
    
    Includes:
    - Objects with bounding boxes and classes
    - Layout elements (walls, floor, ceiling)
    - Doors and windows
    """
    print("Generating scene structure...")
    
    # Build scene structure
    scene = {
        "name": "Reconstructed Living Room",
        "bounds": {
            "min": [-0.1, 0.0, -0.1],
            "max": [5.0, 3.0, 5.0]
        },
        "objects": [],
        "layout": {
            "floor": {"y": 0.0, "material": "wood"},
            "ceiling": {"y": 2.8, "material": "plaster"},
            "walls": [
                {"name": "north", "normal": [0, 0, -1], "position": 5.0, "material": "paint_beige"},
                {"name": "south", "normal": [0, 0, 1], "position": -0.1, "material": "paint_beige"},
                {"name": "east", "normal": [-1, 0, 0], "position": 5.0, "material": "paint_beige"},
                {"name": "west", "normal": [1, 0, 0], "position": -0.1, "material": "paint_beige"},
            ],
            "doors": [
                {"position": [2.5, 1.0, -0.05], "dimensions": [0.9, 2.1], "type": "single"}
            ],
            "windows": [
                {"position": [4.95, 1.5, 2.5], "dimensions": [1.5, 1.2], "type": "double"}
            ]
        }
    }
    
    # Add objects with bounding boxes
    for inst in instances:
        pos = inst["position"]
        dims = inst["dimensions"]
        
        scene["objects"].append({
            "id": inst["id"],
            "class": inst["class"],
            "position": pos,
            "dimensions": dims,
            "bounding_box": {
                "min": [pos[0] - dims[0]/2, pos[1] - dims[1]/2, pos[2] - dims[2]/2],
                "max": [pos[0] + dims[0]/2, pos[1] + dims[1]/2, pos[2] + dims[2]/2]
            },
            "confidence": 0.85 + np.random.random() * 0.1,
            "mask_path": inst.get("mask_path")
        })
    
    # Save scene.json
    scene_path = f"{config.output_dir}/scene.json"
    with open(scene_path, "w") as f:
        json.dump(scene, f, indent=2)
    
    print(f"Saved scene structure to {scene_path}")
    print(f"  - {len(scene['objects'])} objects")
    print(f"  - {len(scene['layout']['walls'])} walls")
    print(f"  - {len(scene['layout']['doors'])} doors")
    print(f"  - {len(scene['layout']['windows'])} windows")
    
    return scene

scene_structure = generate_scene_structure(instances, config)
print("\nScene structure generated!")

## Cell 7: Multimodal QA Function

Function to answer questions about the scene using the LMM.

In [ ]:
from PIL import Image
import torch

# Fallback answers for demo reliability
FALLBACK_ANSWERS = {
    "furniture": "I can see a gray sofa, a wooden coffee table, two armchairs, a floor lamp, a bookshelf, a rug, and a TV stand.",
    "color": "The room features warm neutral tones with beige walls, a gray sofa, and wooden furniture in natural oak finish.",
    "count": "There are 8 objects detected in this scene: 1 sofa, 1 coffee table, 2 armchairs, 1 floor lamp, 1 bookshelf, 1 rug, and 1 TV stand.",
    "chairs": "There are 2 armchairs in the room, positioned on either side of the seating area.",
    "tv": "The TV is mounted above or placed on the TV stand, which is positioned against the south wall facing the sofa.",
    "lighting": "Yes, there is a floor lamp near the corner and natural light coming from the window on the east wall.",
    "default": "This appears to be a modern living room with comfortable seating, good lighting, and functional furniture arrangement."
}

def ask_question(prompt, image_path=None, object_id=None, model=None, processor=None, scene=None):
    """
    Answer a question about the scene.
    
    Args:
        prompt: Question to answer
        image_path: Optional path to image
        object_id: Optional object ID to focus on
        model: LMM model (if None, uses fallback)
        processor: LMM processor
        scene: Scene structure dict
    
    Returns:
        dict with answer, confidence, and metadata
    """
    prompt_lower = prompt.lower()
    
    # Try LMM inference if model is available
    if model is not None and processor is not None:
        try:
            # Prepare input
            messages = [{
                "role": "user",
                "content": []
            }]
            
            # Add image if provided
            if image_path and os.path.exists(image_path):
                image = Image.open(image_path).convert("RGB")
                messages[0]["content"].append({"type": "image", "image": image})
            
            # Add scene context
            if scene:
                context = f"Scene contains: {', '.join([o['class'] for o in scene['objects']])}. "
                messages[0]["content"].append({"type": "text", "text": context + prompt})
            else:
                messages[0]["content"].append({"type": "text", "text": prompt})
            
            # Process and generate
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            
            if image_path and os.path.exists(image_path):
                inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
            else:
                inputs = processor(text=[text], return_tensors="pt").to(model.device)
            
            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=256)
                answer = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            
            return {
                "question": prompt,
                "answer": answer,
                "confidence": 0.85,
                "is_fallback": False
            }
            
        except Exception as e:
            print(f"LMM inference failed: {e}")
            print("Falling back to structured responses...")
    
    # Fallback to keyword-based responses
    if "furniture" in prompt_lower or "object" in prompt_lower:
        answer = FALLBACK_ANSWERS["furniture"]
    elif "color" in prompt_lower or "style" in prompt_lower:
        answer = FALLBACK_ANSWERS["color"]
    elif "how many" in prompt_lower or "count" in prompt_lower:
        answer = FALLBACK_ANSWERS["count"]
    elif "chair" in prompt_lower:
        answer = FALLBACK_ANSWERS["chairs"]
    elif "tv" in prompt_lower or "television" in prompt_lower:
        answer = FALLBACK_ANSWERS["tv"]
    elif "light" in prompt_lower or "lamp" in prompt_lower:
        answer = FALLBACK_ANSWERS["lighting"]
    else:
        answer = FALLBACK_ANSWERS["default"]
    
    return {
        "question": prompt,
        "answer": answer,
        "confidence": 0.90,
        "is_fallback": True
    }

# Test QA
test_questions = [
    "What furniture is in this room?",
    "How many chairs are there?",
    "What is the color scheme?"
]

print("Testing QA function:\n")
for q in test_questions:
    result = ask_question(q, model=lmm_model, processor=lmm_processor, scene=scene_structure)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"   (confidence: {result['confidence']:.2f}, fallback: {result['is_fallback']})")
    print()

## Cell 8: Demo Execution

Run the full pipeline on sample data and test with example questions.

In [ ]:
import time

def run_full_demo(config):
    """Run the complete demo pipeline."""
    print("="*50)
    print("Running Full Demo Pipeline")
    print("="*50)
    
    start_time = time.time()
    
    # Step 1: Load sample images
    print("\n[Step 1/5] Loading sample images...")
    sample_images = list(Path(config.input_dir).glob("*.jpg")) + list(Path(config.input_dir).glob("*.png"))
    if not sample_images:
        print("No images found, using placeholder mode.")
        sample_images = []
    else:
        print(f"Found {len(sample_images)} images.")
    
    # Step 2: Run reconstruction
    print("\n[Step 2/5] Running 3D reconstruction...")
    recon_result = run_reconstruction([str(p) for p in sample_images], config)
    
    # Step 3: Run instance segmentation
    print("\n[Step 3/5] Running instance segmentation...")
    instances = run_instance_segmentation(config)
    
    # Step 4: Generate scene structure
    print("\n[Step 4/5] Generating scene structure...")
    scene = generate_scene_structure(instances, config)
    
    # Step 5: Test QA
    print("\n[Step 5/5] Testing multimodal QA...")
    qa_results = []
    demo_questions = [
        "What furniture is visible in this room?",
        "How many chairs are there?",
        "What is the color scheme of the room?",
        "Where is the TV located?",
        "Is there good lighting in the room?"
    ]
    
    for q in demo_questions:
        result = ask_question(q, model=lmm_model, processor=lmm_processor, scene=scene)
        qa_results.append(result)
        print(f"  Q: {q}")
        print(f"  A: {result['answer'][:100]}..." if len(result['answer']) > 100 else f"  A: {result['answer']}")
    
    elapsed = time.time() - start_time
    
    print("\n" + "="*50)
    print(f"Demo completed in {elapsed:.2f} seconds")
    print("="*50)
    
    return {
        "reconstruction": recon_result,
        "instances": instances,
        "scene": scene,
        "qa_results": qa_results,
        "elapsed_time": elapsed
    }

# Run the demo
demo_results = run_full_demo(config)

## Cell 9: Export Demo Assets

Package outputs into a downloadable zip file for demo use.

In [ ]:
import zipfile
import json
from pathlib import Path

def export_demo_assets(config, demo_results):
    """Package demo outputs into a zip file."""
    print("Exporting demo assets...")
    
    zip_path = f"{config.output_dir}/demo_ready.zip"
    
    # Save answers.json
    answers_path = f"{config.output_dir}/answers.json"
    with open(answers_path, "w") as f:
        json.dump(demo_results["qa_results"], f, indent=2)
    
    # Create zip
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        output_dir = Path(config.output_dir)
        
        # Add main files
        for file_name in ["pointcloud.ply", "scene.json", "poses.json", "answers.json"]:
            file_path = output_dir / file_name
            if file_path.exists():
                zf.write(file_path, file_name)
                print(f"  Added: {file_name}")
        
        # Add instance masks
        instances_dir = output_dir / "instances"
        if instances_dir.exists():
            for mask_file in instances_dir.glob("*.png"):
                zf.write(mask_file, f"instances/{mask_file.name}")
            print(f"  Added: {len(list(instances_dir.glob('*.png')))} instance masks")
        
        # Add sample images if available
        input_dir = Path(config.input_dir)
        images = list(input_dir.glob("*.jpg")) + list(input_dir.glob("*.png"))
        for img in images[:5]:  # Limit to 5 sample images
            zf.write(img, f"samples/{img.name}")
        if images:
            print(f"  Added: {min(len(images), 5)} sample images")
    
    # Get file size
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"\nExport complete: {zip_path}")
    print(f"Archive size: {size_mb:.2f} MB")
    
    return zip_path

# Export assets
zip_file = export_demo_assets(config, demo_results)

# Create download link (for Kaggle/Colab)
try:
    from IPython.display import FileLink, display
    display(FileLink(zip_file))
except:
    print(f"Download manually: {zip_file}")

## Cell 10: API Server (Optional)

Simple Flask server for live demo. Can run in Colab/Kaggle with ngrok.

In [ ]:
# Uncomment and run to start a simple API server

# from flask import Flask, request, jsonify
# import threading

# app = Flask(__name__)

# @app.route('/health', methods=['GET'])
# def health():
#     return jsonify({"status": "healthy", "version": "0.1.0"})

# @app.route('/ask', methods=['POST'])
# def ask():
#     data = request.json
#     prompt = data.get('prompt', '')
#     
#     result = ask_question(
#         prompt,
#         model=lmm_model,
#         processor=lmm_processor,
#         scene=scene_structure
#     )
#     
#     return jsonify(result)

# @app.route('/scene', methods=['GET'])
# def get_scene():
#     return jsonify(scene_structure)

# # Run server in background thread
# def run_server():
#     app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

# server_thread = threading.Thread(target=run_server, daemon=True)
# server_thread.start()
# print("API server started on http://0.0.0.0:5000")
# print("Endpoints: /health, /ask (POST), /scene")

print("API server code ready. Uncomment to run.")
print("For public access, use ngrok or Colab's tunneling.")

## Summary

This notebook provides:

1. **3D Reconstruction Pipeline** (stub) - Generates mock point clouds and camera poses
2. **Instance Segmentation** (stub) - Creates mock object masks and embeddings
3. **Scene Structuring** - Converts outputs to structured scene.json
4. **Multimodal QA** - Answers questions using LMM with fallback support
5. **Demo Export** - Packages all assets for reliable demo delivery

For production use:
- Replace reconstruction stub with IGGT or DUSt3R
- Replace segmentation stub with SAM
- Use higher-quality LMM (7B+) with better quantization
- Add proper error handling and logging